# RAG Exploration & Prototyping 🔬

Use this notebook to experiment with chunk sizes, retrieval strategies, and prompt variations before moving them into the production `app/` folder.

In [ ]:
import os
import sys
from pathlib import Path

# Allow importing from the app/ folder
sys.path.insert(0, str(Path(os.getcwd()).parent))

from dotenv import load_dotenv
load_dotenv(override=True)

## 1. Test Chunking Strategies 📄
Experiment with `RecursiveCharacterTextSplitter` parameters to see how text is split.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

sample_text = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the natural intelligence displayed by animals including humans.
AI research has been defined as the field of study of intelligent agents, which refers to any system that perceives its environment and takes actions that maximize its chance of achieving its goals.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
)

chunks = splitter.split_text(sample_text)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} (Length: {len(chunk)}): {chunk!r}\n")

## 2. Test Vector Retrieval 🔍
Check what documents are returned by ChromaDB for a given query.

In [ ]:
from app.retriever import load_vectorstore, retrieve_with_scores

# Load the existing vector database
vs = load_vectorstore()

question = "When was the company founded?"
results = retrieve_with_scores(question, vectorstore=vs, k=3)

print(f"Question: {question}\n")
for doc, score in results:
    print(f"[Score: {score:.3f}] {doc.page_content}")

## 3. Test Prompt Variations 📝
Draft and test new prompts without altering `app/prompts.py`.

In [ ]:
from langchain_core.prompts import PromptTemplate
from app.chain import build_llm
from app.chain import _format_context

EXPERIMENTAL_PROMPT = PromptTemplate.from_template(
    """
    You are a sarcastic AI assistant. Answer the question using ONLY the provided context.
    If the context doesn't contain the answer, say 'I don't know, ask a human'.
    
    CONTEXT:
    {context}
    
    QUESTION: {question}
    """
)

llm = build_llm()

# Create an experimental chain manually
docs = [doc for doc, _ in results]  # from the previous cell
context_str = _format_context(docs)

chain = EXPERIMENTAL_PROMPT | llm

response = chain.invoke({
    "context": context_str,
    "question": question
})

print(response.content)